# Day 90 — Visualization

This notebook covers Regression Diagnostic Visualization, Train vs. Validation Learning Curves (`plot_learning_curves`), Predicted vs. Actual Calibration Plots, Residual Distribution Analysis (`plot_prediction_diagnostics`), and Matplotlib Memory Hygiene (`plt.close(fig)`)

## Validation goals
- Implement `collect_predictions` using `model.eval()` and `torch.no_grad()` to extract aligned `y_true` and `y_pred` NumPy arrays across evaluation batches
- Build `plot_learning_curves` to track training and validation `RMSE` trajectories across 30 epochs and save `artifacts/day_90_learning_curves.png`
- Build `plot_prediction_diagnostics` to render side-by-side Predicted vs. Actual scatter calibration and Residual Distribution histograms (`artifacts/day_90_diagnostics.png`)
- Analyze visual failure signatures (post-epoch overfitting divergence, high-demand underprediction bends) and enforce Matplotlib figure cleanup via `plt.close(fig)`

### 1. High-Value Plots for Regression Neural Networks

Raw numbers in a terminal log can hide subtle patterns. Three core plots give you most of the diagnostic signal you need for a regression model:
- Train vs. Validation Learning Curves: Plotting training and validation `RMSE` (or `MSE`) across epochs shows whether the model is converging smoothly, underfitting, or overfitting (when validation error levels off or rises while training error keeps dropping).
- Predicted vs. Actual Scatter Plot: Comparing $\hat{y}$ against $y$ alongside a diagonal $45^\circ$ reference line ($y = \hat{y}$) shows overall calibration and whether accuracy holds across low, medium, and high demand levels.
- Residual Distribution & Error vs. Prediction: Plotting residuals ($e_i = y_i - \hat{y}_i$) reveals bias (non-zero mean error), heavy tail errors on demand spikes, or variance growing with larger predicted volumes.

### 2. Implementing Diagnostic Plots (`day_90_visualization.py`)

Below is a clean module that collects predictions and residuals on a DataLoader and saves a multi-panel diagnostic figure (`artifacts/day_90_diagnostics.png`) alongside a training-curve plot (`artifacts/day_90_learning_curves.png`).

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from day_85_dataset import build_dataloaders
from day_86_architecture import RetailDemandMLP
from day_87_training import set_seed
from day_88_validation import fit_with_evaluation


def collect_predictions(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    """Runs inference on a DataLoader and returns (y_true, y_pred) as 1D arrays."""
    model.eval()
    preds_list: list[np.ndarray] = []
    targets_list: list[np.ndarray] = []

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_preds = model(batch_x).squeeze(-1).cpu().numpy()
            batch_targets = batch_y.squeeze(-1).cpu().numpy()

            preds_list.append(batch_preds)
            targets_list.append(batch_targets)

    return np.concatenate(targets_list), np.concatenate(preds_list)


def plot_learning_curves(
    history: list[dict[str, float]],
    output_path: str = "artifacts/day_90_learning_curves.png",
) -> Path:
    """Plots training and validation RMSE across epochs."""
    out_file = Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    epochs = [int(row["epoch"]) for row in history]
    train_rmse = [row["train_rmse"] for row in history]
    val_rmse = [row["val_rmse"] for row in history]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(epochs, train_rmse, marker="o", markersize=4, label="Train RMSE")
    ax.plot(epochs, val_rmse, marker="s", markersize=4, label="Validation RMSE")
    ax.set_title("Training vs. Validation RMSE Across Epochs")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("RMSE (demand_units)")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()

    fig.tight_layout()
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


def plot_prediction_diagnostics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    output_path: str = "artifacts/day_90_diagnostics.png",
) -> Path:
    """Creates side-by-side predicted-vs-actual and residual diagnostic plots."""
    out_file = Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    residuals = y_true - y_pred
    min_val = float(min(y_true.min(), y_pred.min()))
    max_val = float(max(y_true.max(), y_pred.max()))

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))

    # Left panel: Predicted vs. Actual with ideal diagonal line
    axes[0].scatter(y_true, y_pred, alpha=0.65, edgecolors="none", s=28)
    axes[0].plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5, label="Ideal (y_pred = y_true)")
    axes[0].set_title("Predicted vs. Actual Demand")
    axes[0].set_xlabel("Actual demand_units")
    axes[0].set_ylabel("Predicted demand_units")
    axes[0].grid(True, linestyle="--", alpha=0.4)
    axes[0].legend()

    # Right panel: Residual histogram centered at zero error
    axes[1].hist(residuals, bins=25, edgecolor="black", alpha=0.75)
    axes[1].axvline(0.0, color="red", linestyle="--", linewidth=1.5, label="Zero Residual")
    axes[1].axvline(float(np.mean(residuals)), color="orange", linestyle="-", linewidth=1.5, label=f"Mean ({np.mean(residuals):.2f})")
    axes[1].set_title("Residual Distribution (Actual - Predicted)")
    axes[1].set_xlabel("Residual (demand_units)")
    axes[1].set_ylabel("Sample Count")
    axes[1].grid(True, linestyle="--", alpha=0.4)
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    return out_file


if __name__ == "__main__":
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, test_loader, num_features, source = build_dataloaders(batch_size=32)
    model = RetailDemandMLP(
        in_features=num_features,
        hidden_dims=(64, 32),
        dropout=0.1,
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)

    history = fit_with_evaluation(
        model=model,
        train_loader=train_loader,
        eval_loader=test_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=30,
        log_every=10,
    )

    y_true, y_pred = collect_predictions(model=model, dataloader=test_loader, device=device)
    curves_path = plot_learning_curves(history)
    diag_path = plot_prediction_diagnostics(y_true, y_pred)

    print("\n--- Day 90: Diagnostic Plots Saved ---")
    print(f"Data source:       {source}")
    print(f"Learning curves:   {curves_path}")
    print(f"Error diagnostics: {diag_path}")
    print(f"Residual mean:     {np.mean(y_true - y_pred):.3f} units")
    print(f"Residual std:      {np.std(y_true - y_pred):.3f} units")

Epoch  1/30 | Train RMSE: 151.09 | Val RMSE:  84.88 | Val MAE:  81.83 | Val R2: -8.0900
Epoch 10/30 | Train RMSE:  21.83 | Val RMSE:  15.92 | Val MAE:  12.85 | Val R2: 0.6803
Epoch 20/30 | Train RMSE:  21.98 | Val RMSE:  16.69 | Val MAE:  13.71 | Val R2: 0.6484
Epoch 30/30 | Train RMSE:  22.06 | Val RMSE:  15.37 | Val MAE:  12.67 | Val R2: 0.7019

--- Day 90: Diagnostic Plots Saved ---
Data source:       synthetic_fallback
Learning curves:   artifacts\day_90_learning_curves.png
Error diagnostics: artifacts\day_90_diagnostics.png
Residual mean:     2.785 units
Residual std:      15.118 units


### 3. Interpreting What the Charts Tell You

- Points Hugging the Diagonal: In the left scatter plot, tight clustering along the dashed diagonal means predictions track actual demand closely across scales; systematic drift above or below the line signals over- or under-prediction in specific ranges.
- Residual Center and Spread: A residual histogram centered near 0.0 with a narrow spread indicates low bias and consistent error magnitude. Skew or long tails point to unmodeled spikes, missing features, or outlier sensitivity.
- Closing the Figure (`plt.close(fig)`): Always close Matplotlib figures after saving when generating plots inside scripts or loops so memory does not accumulate across runs.

### Guided Question:

If Train `RMSE` continues decreasing steadily after epoch 15 while Validation `RMSE` begins rising, what does that plot tell you about the model's fit?

### Answer:

That divergence after epoch 15 indicates overfitting (rising variance): the model keeps fitting training examples more closely while losing generalization on unseen validation data.

**What the Curve Means**

- Training RMSE dropping: The model is still adjusting parameters to reduce error on the training set, including absorbing sample-specific noise, outliers, or quirks rather than broader signal.
- Validation RMSE rising: Predictions on held-out validation data become less accurate after epoch 15 because weights tailored to training details over-commit to specific patterns that do not hold elsewhere.
- The inflection point (around epoch 15): The minimum of the validation RMSE curve marks the point of best generalization before overfitting accelerates.

**Common Fixes to Test Next**

- Early stopping: Restore the checkpoint from around epoch 15 (where validation RMSE is lowest) or stop training once validation RMSE stops improving for a set patience window.
- Regularization: Increase weight decay / L2 regularization, dropout, or data augmentation if the model has high capacity relative to the training set.
- Capacity or data adjustments: Reduce model size/complexity, add more training data if available, or check whether the validation split has a distribution shift or label noise.

### Guided Question:

On a Predicted vs. Actual scatter plot, what would a noticeable downward bend at high actual demand values suggest about the network's predictions?

### Answer:

A noticeable downward bend (sagging below the 45-degree diagonal reference line) at high actual demand values means the network underpredicts high demand—its predicted values flatten out or lag behind large actual values.

**What This Pattern Usually Indicates**

- Compressed or clipped output range: Bounded output activations (such as sigmoid or tanh on a scaled target), heavy L2 regularization/weight decay, or residual smoothing pull extreme predictions toward the mean or a ceiling.
- Sparse high-demand training examples: High-demand spikes often occur less frequently in the training data, so the loss function's gradient signal is dominated by moderate or typical demand levels.
- Unmodeled surge drivers or non-linearities: Sudden demand spikes may depend on regime shifts, promotions, stock availability, or interaction effects that current features or network capacity do not capture cleanly.
- Target transformation or scale effects: If targets were log-transformed, clipped during preprocessing, or evaluated on a skewed scale, inverse transformations or residual bias can flatten upper-tail predictions.

**Quick Ways to Check the Cause**

- Inspect residuals (`actual - predicted`) plotted against actual demand or predicted demand to confirm whether error grows systematically in the upper tail.
- Check the distribution of actual demand in the training set to see how many high-demand samples exist and whether target clipping occurred.
- Compare linear vs. non-linear or tree-based vs. neural baselines on the same high-demand slice to see whether the flattening comes from feature representation, loss weighting, or model capacity.

### Guided Question:

Why is calling `plt.close(fig)` after `fig.savefig(...)` good practice when generating diagnostic charts programmatically?

### Answer:

Calling `plt.close(fig)` after `fig.savefig(...)` frees the figure's memory and releases pyplot figure state so batch chart generation stays fast, clean, and leak-free.

**Why It Matters in Programmatic Workflows**

- Prevents memory accumulation: `matplotlib.pyplot` keeps references to created figures until explicitly closed. In loops generating dozens or hundreds of diagnostic plots (per epoch, fold, feature, or slice), unclosed figures consume RAM steadily and can slow down runs or trigger out-of-memory errors.
- Avoids state and figure-number collisions: Closing each figure cleanly keeps `pyplot` figure tracking predictable instead of leaving lingering figure objects open in memory across iterations.
- Keeps batch scripts clean: Calling `plt.close(fig)` right after saving ensures each chart is finalized and disposed of before the next plot is constructed.

**Practical Pattern**

- Explicit figure objects: Pair fig, `ax = plt.subplots(...)`, `fig.savefig(...)`, and `plt.close(fig)` together in the same block so cleanup is guaranteed even if you generate many plots in a loop.
- Interactive notebooks vs. scripts: In interactive notebooks, closing a figure immediately after saving is still safe for saved files, though if you also want inline display in the notebook, display or inspect it before closing it.